# Universal 2D to 3D Pipeline

Run the setup cell first, then use the pipeline cell to process images.

In [2]:
# @title 🛠️ Step 1: Environment Setup
# Run this cell ONCE per session to set up dependencies and cache binaries.

import os
import sys
import shutil
import subprocess
import importlib

# --- Setup Options ---
install_texture_support = False # @param {type:"boolean"}
force_reinstall = False # @param {type:"boolean"}

# 1. Mount Google Drive (skip remount if already mounted)
from google.colab import drive
if not os.path.ismount('/content/drive'):
    print("Mounting Google Drive...")
    try:
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Drive mount failed: {e}")
else:
    print("✅ Google Drive already mounted.")

# 2. Shallow-clone Hunyuan3D-2
hunyuan_dir = '/content/Hunyuan3D-2'
if not os.path.exists(hunyuan_dir):
    print("Cloning Hunyuan3D-2 (shallow clone)...")
    subprocess.run([
        "git", "clone", "--depth", "1", "--recurse-submodules",
        "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git", hunyuan_dir
    ], check=True)

# 3. Consolidated, conflict-free Python package installation
setup_marker = '/content/.hy3d_setup_done'
if force_reinstall or not os.path.exists(setup_marker):
    print("Installing Python dependencies (single consolidated pass)...")
    pkgs = [
        "tokenizers==0.21.0",
        "transformers==4.48.2",
        "huggingface_hub>=0.24.0,<1.0",
        "diffusers",
        "accelerate",
        "safetensors",
        "einops",
        "omegaconf",
        "opencv-python",
        "pillow>=10.4.0",
        "numpy<2.5",
        "trimesh",
        "rtree",
        "pymeshlab",
        "pygltflib",
        "xatlas",
        "fast-simplification",
        "rembg",
        "onnxruntime",
        "ninja",
        "pybind11"
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary"] + pkgs, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", hunyuan_dir], check=True)
    with open(setup_marker, 'w') as f:
        f.write('ok')
    print("✅ Python packages installed.")
else:
    print("✅ Python packages already installed (skipping pip).")

# 4. Optional Texture Synthesis (Hunyuan3D-Paint) C++/CUDA Extensions
tex_marker = '/content/.hy3d_tex_setup_done'
if install_texture_support:
    if force_reinstall or not os.path.exists(tex_marker):
        print("Building Hunyuan3D-Paint texture extensions (custom_rasterizer & differentiable_renderer)...")
        subprocess.run(
            [sys.executable, "setup.py", "install"],
            cwd=os.path.join(hunyuan_dir, "hy3dgen/texgen/custom_rasterizer"),
            check=True
        )
        subprocess.run(
            [sys.executable, "setup.py", "install"],
            cwd=os.path.join(hunyuan_dir, "hy3dgen/texgen/differentiable_renderer"),
            check=True
        )
        with open(tex_marker, 'w') as f:
            f.write('ok')
        print("✅ Texture synthesis extensions built.")
    else:
        print("✅ Texture synthesis extensions already built.")

# 5. SolidCraft CLI with Google Drive Binary Caching
solidcraft_dir = "/content/solidcraft"
solidcraft_bin = os.path.join(solidcraft_dir, "target/release/solidcraft-cli")
drive_cache_dir = "/content/drive/MyDrive/.cache/solidcraft"
drive_cached_bin = os.path.join(drive_cache_dir, "solidcraft-cli")

if not force_reinstall and os.path.exists(solidcraft_bin):
    print("✅ SolidCraft binary already ready.")
    if os.path.exists('/content/drive/MyDrive') and not os.path.exists(drive_cached_bin):
        os.makedirs(drive_cache_dir, exist_ok=True)
        shutil.copy2(solidcraft_bin, drive_cached_bin)
        print(f"💾 Cached SolidCraft binary to {drive_cached_bin} for instant future startups.")
elif not force_reinstall and os.path.exists(drive_cached_bin):
    print("⚡ Restoring pre-built SolidCraft binary from Google Drive cache...")
    os.makedirs(os.path.dirname(solidcraft_bin), exist_ok=True)
    shutil.copy2(drive_cached_bin, solidcraft_bin)
    os.chmod(solidcraft_bin, 0o755)
    print("✅ SolidCraft binary restored from Drive in <1s!")
else:
    cargo_bin = os.path.expanduser('~/.cargo/bin/cargo')
    if not os.path.exists(cargo_bin):
        print("Installing Rust toolchain...")
        subprocess.run("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    os.environ["PATH"] += f":{os.path.expanduser('~')}/.cargo/bin"

    if not os.path.exists(solidcraft_dir):
        print("Cloning SolidCraft repository (shallow clone)...")
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/tangency-labs/solidcraft.git", solidcraft_dir
        ], check=True)

    print("Building SolidCraft release binary (optimized, minimal features)...")
    subprocess.run(
        ["cargo", "build", "--release", "--no-default-features", "-p", "solidcraft-cli"],
        cwd=solidcraft_dir,
        check=True
    )
    if os.path.exists('/content/drive/MyDrive'):
        os.makedirs(drive_cache_dir, exist_ok=True)
        shutil.copy2(solidcraft_bin, drive_cached_bin)
        print(f"💾 Cached SolidCraft binary to {drive_cached_bin} for instant future startups.")

# Refresh module import caches without killing the Colab kernel
importlib.invalidate_caches()
if hunyuan_dir not in sys.path:
    sys.path.insert(0, hunyuan_dir)

print("\n✅ Environment setup complete! You can now run Step 2.")

✅ Google Drive already mounted.
✅ Python packages already installed (skipping pip).
✅ SolidCraft binary already ready.
💾 Cached SolidCraft binary to /content/drive/MyDrive/.cache/solidcraft/solidcraft-cli for instant future startups.

✅ Environment setup complete! You can now run Step 2.


In [3]:
# @title 🚀 Step 2: 2D Image to 3D Model Pipeline
# Modify the settings below and run the cell to generate a print-ready 3D model.

import os
import sys
import subprocess
import PIL.Image
import numpy as np
import torch

# --- 1. Project & File Settings ---
project_folder = "3D Print Projects" # @param {type:"string"}
input_image_name = "kittylipglossholder.png" # @param {type:"string"}
output_model_name = "kittylipglossholder_model" # @param {type:"string"}
export_only_raw_obj = False # @param {type:"boolean"}

# --- 2. 3D Quality & Speed Settings ---
remove_background = True # @param {type:"boolean"}
use_flashvdm_turbo = True # @param {type:"boolean"}
octree_resolution = 384 # @param [256, 384, 512] {type:"raw"}
num_inference_steps = 50 # @param {type:"slider", min:20, max:100, step:5}
guidance_scale = 5.5 # @param {type:"number"}
seed = 42 # @param {type:"integer"}

# --- 3. Mesh Repair, Print Scaling & Optional Texture ---
max_faces = 50000 # @param {type:"integer"}
target_size_mm = 100.0 # @param {type:"number"}
enable_texture_synthesis = False # @param {type:"boolean"}

drive_base = "/content/drive/MyDrive"
project_dir = os.path.join(drive_base, project_folder)
os.makedirs(project_dir, exist_ok=True)

input_image_path = os.path.join(project_dir, input_image_name)
raw_obj_path = os.path.join(project_dir, f"{output_model_name}_raw.obj")
clean_obj_path = os.path.join(project_dir, f"{output_model_name}_clean.obj")
textured_glb_path = os.path.join(project_dir, f"{output_model_name}_textured.glb")
final_output_path = os.path.join(project_dir, f"{output_model_name}_print_ready.3mf")

if not os.path.exists(input_image_path):
    raise FileNotFoundError(
        f"❌ Input image not found at '{input_image_path}'. Please upload '{input_image_name}' to '{project_dir}' in Google Drive."
    )

print("=== STEP 1: Image Pre-processing & Hunyuan3D Synthesis ===")
if not hasattr(PIL.Image, "ANTIALIAS"):
    PIL.Image.ANTIALIAS = PIL.Image.LANCZOS
repo_path = "/content/Hunyuan3D-2"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Load and optionally isolate foreground object
input_img = PIL.Image.open(input_image_path)
if remove_background:
    from hy3dgen.rembg import BackgroundRemover
    if 'hy3d_bg_remover' not in globals():
        print("Initializing background remover (rembg)...")
        global hy3d_bg_remover
        hy3d_bg_remover = BackgroundRemover()
    print("Removing background for cleaner 3D geometry...")
    cond_image = hy3d_bg_remover(input_img.convert("RGBA"))
else:
    cond_image = input_img.convert("RGBA")

from hy3dgen.shapegen import (
    Hunyuan3DDiTFlowMatchingPipeline,
    FloaterRemover,
    DegenerateFaceRemover,
    FaceReducer,
)

if 'hunyuan_pipeline' not in globals():
    print("Loading Hunyuan3D-2 pipeline (cached in GPU memory for subsequent runs)...")
    global hunyuan_pipeline, _cached_flashvdm_state
    hunyuan_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
        "tencent/Hunyuan3D-2", torch_dtype=torch.float16, device="cuda"
    )
    _cached_flashvdm_state = False
else:
    print("Using cached Hunyuan3D-2 pipeline.")

if globals().get('_cached_flashvdm_state', False) != use_flashvdm_turbo:
    print(f"Configuring FlashVDM Turbo decoder (enabled={use_flashvdm_turbo})...")
    hunyuan_pipeline.enable_flashvdm(enabled=use_flashvdm_turbo)
    _cached_flashvdm_state = use_flashvdm_turbo

generator = torch.Generator(device="cuda").manual_seed(int(seed)) if seed >= 0 else None
print(
    f"Synthesizing 3D mesh for '{input_image_name}' "
    f"(octree={octree_resolution}, steps={num_inference_steps}, cfg={guidance_scale}, seed={seed})..."
)
raw_synthesis_mesh = hunyuan_pipeline(
    image=cond_image,
    num_inference_steps=int(num_inference_steps),
    guidance_scale=float(guidance_scale),
    octree_resolution=int(octree_resolution),
    generator=generator,
)[0]
raw_synthesis_mesh.export(raw_obj_path)
print(f"✅ Raw mesh saved to {raw_obj_path} ({len(raw_synthesis_mesh.faces):,} faces)")

if export_only_raw_obj:
    print("\n✅ Process complete! Exported raw OBJ only as requested.")
else:
    print("\n=== STEP 2: Topology-Preserving Mesh Cleaning, Repair & Scaling ===")
    import trimesh
    mesh = raw_synthesis_mesh.copy()
    if isinstance(mesh, trimesh.Scene):
        mesh = mesh.dump(concatenate=True)
    elif isinstance(mesh, (list, tuple)):
        mesh = trimesh.util.concatenate(mesh)

    print("Removing disconnected floaters and degenerate faces...")
    mesh = FloaterRemover()(mesh)
    mesh = DegenerateFaceRemover()(mesh)

    if max_faces > 0 and len(mesh.faces) > max_faces:
        print(f"Decimating mesh from {len(mesh.faces):,} to {max_faces:,} faces (preserving boundaries & normals)...")
        mesh = FaceReducer()(mesh, max_facenum=int(max_faces))

    # Final manifold & watertight repair
    mesh.merge_vertices()
    mesh.remove_unreferenced_vertices()
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    trimesh.repair.fix_inversion(mesh)

    # Scale to real-world millimeters and place flat on Z=0 build plate
    if target_size_mm and target_size_mm > 0:
        extents = mesh.extents
        max_extent = float(np.max(extents))
        if max_extent > 0:
            scale_factor = float(target_size_mm) / max_extent
            mesh.apply_scale(scale_factor)
            # Center X/Y at origin and rest bottom on Z = 0
            bounds = mesh.bounds
            center_xy = (bounds[0][:2] + bounds[1][:2]) / 2.0
            min_z = bounds[0][2]
            mesh.apply_translation([-center_xy[0], -center_xy[1], -min_z])
            print(
                f"📏 Scaled model to {mesh.extents[0]:.1f} × {mesh.extents[1]:.1f} × {mesh.extents[2]:.1f} mm "
                f"(placed on Z=0 build plate)"
            )

    mesh.export(clean_obj_path)
    print(f"✅ Cleaned mesh saved to {clean_obj_path} (Faces: {len(mesh.faces):,}, Watertight: {mesh.is_watertight})")

    # Optional Step 2B: Texture Synthesis (Hunyuan3D-Paint)
    if enable_texture_synthesis:
        print("\n=== STEP 2B: Optional Texture Synthesis (Hunyuan3D-Paint) ===")
        try:
            from hy3dgen.texgen import Hunyuan3DPaintPipeline
            if 'hunyuan_tex_pipeline' not in globals():
                print("Loading Hunyuan3D-Paint Turbo pipeline...")
                global hunyuan_tex_pipeline
                hunyuan_tex_pipeline = Hunyuan3DPaintPipeline.from_pretrained(
                    "tencent/Hunyuan3D-2", subfolder="hunyuan3d-paint-v2-0-turbo"
                )
            textured_mesh = hunyuan_tex_pipeline(mesh.copy(), image=cond_image)
            textured_mesh.export(textured_glb_path)
            print(f"🎨 Textured 3D model saved to: {textured_glb_path}")
        except Exception as tex_err:
            print(
                f"⚠️ Texture synthesis skipped/failed ({tex_err}). "
                f"Make sure 'install_texture_support' is checked in Step 1."
            )

    print("\n=== STEP 3: Processing through SolidCraft ===")
    os.environ["PATH"] += f":{os.path.expanduser('~')}/.cargo/bin"
    solidcraft_cli = "/content/solidcraft/target/release/solidcraft-cli"

    cmd = [
        solidcraft_cli,
        "run",
        "--in", clean_obj_path,
        "--out", final_output_path,
        "--quality", "fine",
        "--quiet"
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and os.path.exists(final_output_path):
        print(f"\n✅ Success! Print-ready 3MF file generated at: {final_output_path}")
        print(f"📦 File size: {os.path.getsize(final_output_path) / (1024 * 1024):.2f} MB")
    else:
        print("❌ SolidCraft CLI execution failed.")
        if result.stdout:
            print("Stdout:", result.stdout)
        if result.stderr:
            print("Stderr:", result.stderr)

=== STEP 1: Image Pre-processing & Hunyuan3D Synthesis ===


Initializing background remover (rembg)...


  0%|                                              | 0.00/1.02G [00:00<?, ?B/s]

Removing background for cleaner 3D geometry...


2026-11-16 04:28:10,462 - hy3dgen.shapgen - INFO - Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
INFO:hy3dgen.shapgen:Try to load model from local path: /root/.cache/hy3dgen/tencent/Hunyuan3D-2/hunyuan3d-dit-v2-0
2026-11-16 04:28:10,463 - hy3dgen.shapgen - INFO - Model path not exists, try to download from huggingface
INFO:hy3dgen.shapgen:Model path not exists, try to download from huggingface


Loading Hunyuan3D-2 pipeline (cached in GPU memory for subsequent runs)...


... (12 outputs left out)

2026-11-16 04:29:19,945 - hy3dgen.shapgen - INFO - Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-vae-v2-0-turbo/model.fp16.safetensors
INFO:hy3dgen.shapgen:Loading model from /root/.cache/huggingface/hub/models--tencent--Hunyuan3D-2/snapshots/9cd649ba6913f7a852e3286bad86bfa9a2d83dcf/hunyuan3d-vae-v2-0-turbo/model.fp16.safetensors


PointCrossAttentionEncoder INFO: pc_sharpedge_size is given, using pc_size=5120, pc_sharpedge_size=5120
Synthesizing 3D mesh for 'kittylipglossholder.png' (octree=384, steps=50, cfg=5.5, seed=42)...


Diffusion Sampling:: 100%|██████████| 50/50 [01:03<00:00,  1.28s/it]
2026-11-16 04:30:30,104 - hy3dgen.shapgen - INFO - FlashVDMVolumeDecoding Resolution: [95, 190, 380]
INFO:hy3dgen.shapgen:FlashVDMVolumeDecoding Resolution: [95, 190, 380]
FlashVDM Volume Decoding: 100%|██████████| 64/64 [00:00<00:00, 237.82it/s]
/content/Hunyuan3D-2/hy3dgen/shapegen/models/autoencoders/volume_decoders.py:81: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:356.)
  sliced = padded[slice_dims]


✅ Raw mesh saved to /content/drive/MyDrive/3D Print Projects/kittylipglossholder_model_raw.obj (782,284 faces)

=== STEP 2: Topology-Preserving Mesh Cleaning, Repair & Scaling ===
Removing disconnected floaters and degenerate faces...
Decimating mesh from 782,282 to 50,000 faces (preserving boundaries & normals)...
📏 Scaled model to 63.3 × 100.0 × 57.7 mm (placed on Z=0 build plate)
✅ Cleaned mesh saved to /content/drive/MyDrive/3D Print Projects/kittylipglossholder_model_clean.obj (Faces: 50,000, Watertight: True)

=== STEP 3: Processing through SolidCraft ===

✅ Success! Print-ready 3MF file generated at: /content/drive/MyDrive/3D Print Projects/kittylipglossholder_model_print_ready.3mf
📦 File size: 0.71 MB
